# BraTS 2024 — ABMIL + U-Net Pipeline

## How this notebook works (Simple Explanation)

```
Brain MRI Scan (128, 128, 128, 4)
         ↓
  STEP 1: Cut into 64 small patches
         ↓
  STEP 2: ABMIL looks at each patch
          → Finds which patches have tumor
          → Ignores healthy patches
         ↓
  STEP 3: Only suspicious patches go to U-Net
         ↓
  STEP 4: U-Net draws exact tumor boundary
```

## Cell 1 — Imports

In [ ]:
import os
import numpy as np
import nibabel as nib
import tensorflow as tf
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.utils import shuffle
from tensorflow.keras.utils import to_categorical
from matplotlib import pyplot as plt

print('TensorFlow:', tf.__version__)
print('PyTorch:   ', torch.__version__)

## Cell 2 — Load NIfTI Files (Same as your original)

In [ ]:
def load_nii(path, is_mask=False):
    """
    Load a NIfTI (.nii) file and return it as a NumPy array.
    
    Input  → file path (string)
    Output → numpy array of shape (128, 128, 128)
    """
    nii = nib.load(path)
    arr = np.array(nii.get_fdata(), dtype=np.float32)
    if is_mask:
        arr = arr.astype(np.int32)
    return arr

## Cell 3 — Patch Extraction

### What this does (simple):
Think of the brain MRI like a **Rubik's cube (128×128×128)**.
We cut it into **64 smaller cubes (32×32×32)** — these are the patches.

```
Input:  (128, 128, 128, 4)   ← full brain scan, 4 MRI types
Output: (64,  32,  32,  32, 4) ← 64 small cubes, each 32×32×32, 4 MRI types
```

In [ ]:
def extract_patches(volume, mask, patch_size=32):
    """
    Cut the brain MRI volume into small 3D patches.

    Input:
        volume     → shape (128, 128, 128, 4)  — 4 MRI modalities stacked
        mask       → shape (128, 128, 128, 5)  — segmentation labels
        patch_size → size of each cube (default 32)

    Output:
        patches      → shape (64, 32, 32, 32, 4)  — 64 patches, each with 4 channels
        mask_patches → shape (64, 32, 32, 32, 5)  — matching mask patches
    """
    D, H, W, C = volume.shape          # (128, 128, 128, 4)
    P = patch_size                     # 32

    patches      = []
    mask_patches = []

    # Slide a 32×32×32 window across the full volume
    # Steps: 0, 32, 64, 96  →  4 steps per axis  →  4×4×4 = 64 patches total
    for d in range(0, D, P):
        for h in range(0, H, P):
            for w in range(0, W, P):
                patch      = volume[d:d+P, h:h+P, w:w+P, :]   # (32, 32, 32, 4)
                mask_patch = mask[d:d+P,   h:h+P, w:w+P, :]   # (32, 32, 32, 5)

                patches.append(patch)
                mask_patches.append(mask_patch)

    patches      = np.array(patches)       # (64, 32, 32, 32, 4)
    mask_patches = np.array(mask_patches)  # (64, 32, 32, 32, 5)

    return patches, mask_patches


# Quick shape check
dummy_volume = np.random.rand(128, 128, 128, 4).astype(np.float32)
dummy_mask   = np.random.rand(128, 128, 128, 5).astype(np.float32)
p, m = extract_patches(dummy_volume, dummy_mask)
print('Patches shape:      ', p.shape)   # (64, 32, 32, 32, 4)
print('Mask patches shape: ', m.shape)   # (64, 32, 32, 32, 5)

## Cell 4 — ABMIL Model (PyTorch)

### What this does (simple):
ABMIL is the **smart detective** that looks at all 64 patches and decides:
- Patch 5  → weight 0.9  (very suspicious — tumor here!)
- Patch 12 → weight 0.8  (suspicious)
- Patch 1  → weight 0.01 (boring healthy tissue, ignore)

```
Input:  (64, 32, 32, 32, 4)   ← 64 patches
Output: weights (64, 1)        ← importance score for each patch
```

In [ ]:
# -----------------------------------------------
# Part A: CNN that looks at one patch
#         Input:  (batch, 4, 32, 32, 32)
#         Output: (batch, 128)  ← 128 feature numbers
# -----------------------------------------------
class PatchFeatureExtractor(nn.Module):
    def __init__(self, in_channels=4, out_dim=128):
        super().__init__()
        self.encoder = nn.Sequential(
            # Layer 1: sees basic shapes and edges
            nn.Conv3d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU(),
            nn.MaxPool3d(2),          # 32×32×32 → 16×16×16

            # Layer 2: sees more complex patterns
            nn.Conv3d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(),
            nn.MaxPool3d(2),          # 16×16×16 → 8×8×8

            # Layer 3: sees high-level tumor features
            nn.Conv3d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm3d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool3d(1)   # 8×8×8 → 1×1×1  (any size input works)
        )
        self.fc = nn.Linear(128, out_dim)

    def forward(self, x):
        # x: (batch, 4, 32, 32, 32)
        x = self.encoder(x)           # → (batch, 128, 1, 1, 1)
        x = x.view(x.size(0), -1)     # → (batch, 128)
        return self.fc(x)             # → (batch, 128)


# -----------------------------------------------
# Part B: Attention scorer — which patch is important?
#         Input:  (1, 64, 128)  ← 64 patches, each 128 features
#         Output: (1, 64, 1)    ← importance score per patch
# -----------------------------------------------
class PatchAttention(nn.Module):
    def __init__(self, feature_dim=128):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(feature_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

    def forward(self, patch_features):
        # patch_features: (1, 64, 128)
        scores  = self.attention(patch_features)          # (1, 64, 1)
        weights = torch.softmax(scores, dim=1)            # sum to 1 across 64 patches
        bag_feature = torch.sum(weights * patch_features, dim=1)  # (1, 128)
        return bag_feature, weights


# -----------------------------------------------
# Part C: Full ABMIL Model
#         Input:  (64, 4, 32, 32, 32)  ← 64 patches
#         Output: weights (64, 1)       ← which patches matter
# -----------------------------------------------
class ABMIL(nn.Module):
    def __init__(self, feature_dim=128):
        super().__init__()
        self.feature_extractor = PatchFeatureExtractor(in_channels=4, out_dim=feature_dim)
        self.patch_attention    = PatchAttention(feature_dim)

    def forward(self, patches):
        """
        patches: (N, 4, 32, 32, 32)  ← N patches (e.g. 64)
        """
        N = patches.shape[0]

        # Step 1: Extract features from every patch
        feats = self.feature_extractor(patches)    # (64, 128)
        feats = feats.unsqueeze(0)                 # (1, 64, 128) — add batch dim

        # Step 2: Score each patch by importance
        bag_feature, weights = self.patch_attention(feats)  # weights: (1, 64, 1)

        weights = weights.squeeze(0).squeeze(-1)   # (64,) — one score per patch
        return weights


print('ABMIL model defined successfully.')

## Cell 5 — Patch Selector

### What this does (simple):
ABMIL gives each patch a score. We keep only the **top patches** (highest scores).
Like a detective saying: *"Only investigate the top 10 most suspicious rooms."*

```
Input:  64 patches + their scores
Output: top 10 patches (the suspicious ones)
```

In [ ]:
def select_top_patches(patches, mask_patches, abmil_model, top_k=10, device='cpu'):
    """
    Use ABMIL to find the most suspicious patches.

    Input:
        patches      → (64, 32, 32, 32, 4)   numpy
        mask_patches → (64, 32, 32, 32, 5)   numpy
        abmil_model  → trained ABMIL (PyTorch)
        top_k        → how many top patches to keep (default 10)

    Output:
        top_patches      → (top_k, 32, 32, 32, 4)   numpy
        top_mask_patches → (top_k, 32, 32, 32, 5)   numpy
        top_indices      → which patch numbers were selected
        weights          → attention score for every patch
    """
    abmil_model.eval()

    # Convert patches to PyTorch format
    # NumPy:   (64, 32, 32, 32, 4)  →  channels last
    # PyTorch: (64,  4, 32, 32, 32) →  channels first (required by Conv3d)
    patches_torch = torch.tensor(
        patches.transpose(0, 4, 1, 2, 3),   # move channel axis
        dtype=torch.float32
    ).to(device)

    with torch.no_grad():
        weights = abmil_model(patches_torch)   # (64,) — one score per patch

    weights_np = weights.cpu().numpy()         # back to numpy

    # Sort patches by score — highest first
    top_indices = np.argsort(weights_np)[::-1][:top_k]   # top_k indices

    top_patches      = patches[top_indices]        # (top_k, 32, 32, 32, 4)
    top_mask_patches = mask_patches[top_indices]   # (top_k, 32, 32, 32, 5)

    print(f'Total patches:       {len(patches)}')
    print(f'Top patches kept:    {top_k}')
    print(f'Selected patch IDs:  {sorted(top_indices)}')
    print(f'Top patches shape:   {top_patches.shape}')

    return top_patches, top_mask_patches, top_indices, weights_np


print('Patch selector defined.')

## Cell 6 — U-Net Model (TensorFlow/Keras)

### What this does (simple):
U-Net takes one suspicious patch **(32×32×32×4)** and draws exactly **where the tumor is** inside it.

```
Input:  (batch, 32, 32, 32, 4)   ← suspicious patches
Output: (batch, 32, 32, 32, 5)   ← tumor map (5 classes)
```

In [ ]:
from tensorflow.keras.layers import (
    Input, Conv3D, MaxPooling3D, UpSampling3D,
    concatenate, BatchNormalization, Activation, Dropout
)
from tensorflow.keras.models import Model


def conv_block(x, filters):
    """
    Two convolutions with BatchNorm and ReLU.
    This is the basic building block of U-Net.
    """
    x = Conv3D(filters, kernel_size=3, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv3D(filters, kernel_size=3, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    return x


def build_patch_unet(patch_size=32, channels=4, num_classes=5):
    """
    Build a 3D U-Net for patch-level segmentation.

    Input:  (batch, 32, 32, 32, 4)
    Output: (batch, 32, 32, 32, 5)

    U-Net shape (like the letter U):

    Input
      ↓ Encoder (going down — learning features)
    [32 filters] → [64 filters] → [128 filters] → [256 filters]
      ↓ Bottleneck (deepest understanding)
    [512 filters]
      ↓ Decoder (going up — drawing the tumor map)
    [256 filters] → [128 filters] → [64 filters] → [32 filters]
      ↓
    Output (5 classes)
    """
    inputs = Input(shape=(patch_size, patch_size, patch_size, channels))

    # ---- ENCODER (going down) ----
    # Each level: learn features, then shrink the size
    c1 = conv_block(inputs, 32)           # (32, 32, 32, 32)
    p1 = MaxPooling3D(pool_size=2)(c1)    # (16, 16, 16, 32)

    c2 = conv_block(p1, 64)               # (16, 16, 16, 64)
    p2 = MaxPooling3D(pool_size=2)(c2)    # (8,  8,  8,  64)

    c3 = conv_block(p2, 128)              # (8,  8,  8,  128)
    p3 = MaxPooling3D(pool_size=2)(c3)    # (4,  4,  4,  128)

    # ---- BOTTLENECK (deepest level) ----
    c4 = conv_block(p3, 256)              # (4,  4,  4,  256)
    c4 = Dropout(0.3)(c4)

    # ---- DECODER (going up) ----
    # Each level: grow the size back, combine with encoder features
    # The "skip connections" (concatenate) pass encoder info directly
    # to the decoder — this is what makes U-Net special
    u5 = UpSampling3D(size=2)(c4)         # (8,  8,  8,  256)
    u5 = concatenate([u5, c3])            # combine with encoder level 3
    c5 = conv_block(u5, 128)

    u6 = UpSampling3D(size=2)(c5)         # (16, 16, 16, 128)
    u6 = concatenate([u6, c2])            # combine with encoder level 2
    c6 = conv_block(u6, 64)

    u7 = UpSampling3D(size=2)(c6)         # (32, 32, 32, 64)
    u7 = concatenate([u7, c1])            # combine with encoder level 1
    c7 = conv_block(u7, 32)

    # ---- OUTPUT ----
    outputs = Conv3D(num_classes, kernel_size=1, activation='softmax')(c7)
    # Shape: (batch, 32, 32, 32, 5)

    model = Model(inputs, outputs, name='PatchUNet')
    return model


# Build and summarize
patch_unet = build_patch_unet(patch_size=32, channels=4, num_classes=5)
print('Input  shape:', patch_unet.input_shape)   # (None, 32, 32, 32, 4)
print('Output shape:', patch_unet.output_shape)  # (None, 32, 32, 32, 5)

## Cell 7 — Loss Functions and Metrics (Same as your original)

In [ ]:
import tensorflow as tf

# ---- Dice Coefficient ----
def dice_coef(y_true, y_pred, smooth=1.0):
    y_true_f = tf.keras.backend.flatten(tf.cast(y_true, tf.float32))
    y_pred_f = tf.keras.backend.flatten(tf.cast(y_pred, tf.float32))
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (
        tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth
    )

# ---- Dice Loss ----
def dice_loss(y_true, y_pred):
    return 1 - dice_coef(y_true, y_pred)

# ---- Combined Loss: Dice + CrossEntropy ----
def total_loss(y_true, y_pred):
    return dice_loss(y_true, y_pred) + \
           tf.keras.losses.categorical_crossentropy(y_true, y_pred)


# ---- Per-Class Dice Metric ----
class DicePerClass(tf.keras.metrics.Metric):
    def __init__(self, class_index, name=None, smooth=1.0, **kwargs):
        if name is None:
            name = f'dice_class_{class_index}'
        super().__init__(name=name, **kwargs)
        self.class_index = class_index
        self.smooth = smooth
        self.inter = self.add_weight(name='inter', initializer='zeros')
        self.den   = self.add_weight(name='den',   initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        yt = tf.cast(y_true[..., self.class_index], tf.float32)
        yp = tf.cast(y_pred[..., self.class_index], tf.float32)
        self.inter.assign_add(tf.reduce_sum(yt * yp))
        self.den.assign_add(tf.reduce_sum(yt + yp))

    def result(self):
        return (2. * self.inter + self.smooth) / (self.den + self.smooth)

    def reset_state(self):
        self.inter.assign(0.0)
        self.den.assign(0.0)


# ---- Whole Tumor Dice ----
class DiceWholeTumorMetric(tf.keras.metrics.Metric):
    def __init__(self, name='dice_whole_tumor', smooth=1.0, **kwargs):
        super().__init__(name=name, **kwargs)
        self.smooth = smooth
        self.inter = self.add_weight(name='inter', initializer='zeros')
        self.den   = self.add_weight(name='den',   initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        yt = tf.cast(y_true[...,1] + y_true[...,2] + y_true[...,3], tf.float32)
        yp = tf.cast(y_pred[...,1] + y_pred[...,2] + y_pred[...,3], tf.float32)
        self.inter.assign_add(tf.reduce_sum(yt * yp))
        self.den.assign_add(tf.reduce_sum(yt + yp))

    def result(self):
        return (2. * self.inter + self.smooth) / (self.den + self.smooth)

    def reset_state(self):
        self.inter.assign(0.0)
        self.den.assign(0.0)


print('Loss functions and metrics defined.')

## Cell 8 — Data Generator (ABMIL + U-Net combined)

### What this does (simple):
This is the **feeding system** for training.
For each patient it:
1. Loads the MRI scan
2. Cuts it into 64 patches
3. ABMIL finds the top suspicious patches
4. Those suspicious patches are fed to U-Net for training

```
For each patient:
  Load scan  →  Extract 64 patches  →  ABMIL selects top 10  →  U-Net trains on top 10
```

In [ ]:
def imageLoader_abmil_unet(
    root_dir,
    patient_list,
    abmil_model,
    batch_size   = 1,
    patch_size   = 32,
    top_k        = 10,
    device       = 'cpu'
):
    """
    Data generator that combines ABMIL patch selection with U-Net training.

    Yields batches of:
        X → (batch_size * top_k, 32, 32, 32, 4)  ← top suspicious patches
        Y → (batch_size * top_k, 32, 32, 32, 5)  ← matching masks
    """
    L       = len(patient_list)
    indices = np.arange(L)

    while True:
        indices = shuffle(indices, random_state=None)

        batch_start = 0
        batch_end   = batch_size

        while batch_start < L:
            limit         = min(batch_end, L)
            batch_indices = indices[batch_start:limit]

            X_batch, Y_batch = [], []

            for i in batch_indices:
                patient_id   = patient_list[i]
                patient_path = os.path.join(root_dir, patient_id)

                # ---- Check all files exist ----
                files_needed = [
                    f'{patient_id}-t1n.nii',
                    f'{patient_id}-t1c.nii',
                    f'{patient_id}-t2w.nii',
                    f'{patient_id}-t2f.nii',
                    f'{patient_id}-seg.nii'
                ]
                if not all(os.path.exists(os.path.join(patient_path, f)) for f in files_needed):
                    print(f'Skipping {patient_id} — missing files')
                    continue

                # ---- Load 4 MRI modalities ----
                t1c  = load_nii(os.path.join(patient_path, f'{patient_id}-t1c.nii'))
                t1n  = load_nii(os.path.join(patient_path, f'{patient_id}-t1n.nii'))
                t2f  = load_nii(os.path.join(patient_path, f'{patient_id}-t2f.nii'))
                t2w  = load_nii(os.path.join(patient_path, f'{patient_id}-t2w.nii'))
                mask = load_nii(os.path.join(patient_path, f'{patient_id}-seg.nii'), is_mask=True)

                # ---- Normalize each modality ----
                def norm(x):
                    return (x - np.mean(x)) / (np.std(x) + 1e-8)

                t1c = norm(t1c)
                t1n = norm(t1n)
                t2f = norm(t2f)
                t2w = norm(t2w)

                # ---- Stack 4 modalities → (128, 128, 128, 4) ----
                volume = np.stack([t1c, t1n, t2f, t2w], axis=-1)

                # ---- One-hot encode mask → (128, 128, 128, 5) ----
                mask_cat = to_categorical(mask, num_classes=5)

                # ---- STEP 1: Cut into 64 patches ----
                patches, mask_patches = extract_patches(volume, mask_cat, patch_size)
                # patches:      (64, 32, 32, 32, 4)
                # mask_patches: (64, 32, 32, 32, 5)

                # ---- STEP 2: ABMIL finds top suspicious patches ----
                top_patches, top_masks, _, _ = select_top_patches(
                    patches, mask_patches, abmil_model, top_k=top_k, device=device
                )
                # top_patches: (10, 32, 32, 32, 4)
                # top_masks:   (10, 32, 32, 32, 5)

                X_batch.append(top_patches)
                Y_batch.append(top_masks)

            if len(X_batch) == 0:
                batch_start += batch_size
                batch_end   += batch_size
                continue

            # Stack all patients in the batch
            X = np.concatenate(X_batch, axis=0).astype(np.float32)
            Y = np.concatenate(Y_batch, axis=0).astype(np.float32)
            # X: (batch_size * top_k, 32, 32, 32, 4)
            # Y: (batch_size * top_k, 32, 32, 32, 5)

            yield (X, Y)

            batch_start += batch_size
            batch_end   += batch_size


print('Data generator defined.')

## Cell 9 — Setup Paths and Data Splits (Same as your original)

In [ ]:
# ---- Dataset Path ----
root_dir = '/kaggle/input/datasets/muhammadwahajsajid/brats-2024-dataset-full-tumor-centered-cropped/Training/Training'

# ---- Get all patient IDs ----
all_patients = sorted(os.listdir(root_dir))
print(f'Total patients: {len(all_patients)}')

# ---- Train / Validation Split (80/20) ----
split_idx          = int(len(all_patients) * 0.8)
train_patient_list = all_patients[:split_idx]
val_patient_list   = all_patients[split_idx:]

print(f'Training patients:   {len(train_patient_list)}')
print(f'Validation patients: {len(val_patient_list)}')

## Cell 10 — Initialize ABMIL and Create Generators

In [ ]:
# ---- Hyperparameters ----
BATCH_SIZE = 1     # 1 patient at a time (memory safe for 3D MRI)
PATCH_SIZE = 32    # each patch is 32×32×32
TOP_K      = 10    # keep top 10 most suspicious patches per patient
EPOCHS     = 85
LR         = 1e-3
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

# ---- Initialize ABMIL (untrained — will learn during training) ----
abmil_model = ABMIL(feature_dim=128).to(DEVICE)
print('ABMIL initialized.')

# ---- Create Data Generators ----
train_generator = imageLoader_abmil_unet(
    root_dir      = root_dir,
    patient_list  = train_patient_list,
    abmil_model   = abmil_model,
    batch_size    = BATCH_SIZE,
    patch_size    = PATCH_SIZE,
    top_k         = TOP_K,
    device        = DEVICE
)

val_generator = imageLoader_abmil_unet(
    root_dir      = root_dir,
    patient_list  = val_patient_list,
    abmil_model   = abmil_model,
    batch_size    = BATCH_SIZE,
    patch_size    = PATCH_SIZE,
    top_k         = TOP_K,
    device        = DEVICE
)

# ---- Quick shape test ----
test_x, test_y = next(train_generator)
print(f'Generator X shape: {test_x.shape}')   # (10, 32, 32, 32, 4)
print(f'Generator Y shape: {test_y.shape}')   # (10, 32, 32, 32, 5)

## Cell 11 — Compile and Train U-Net

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
)

# ---- Build U-Net for 32×32×32 patches ----
patch_unet = build_patch_unet(patch_size=PATCH_SIZE, channels=4, num_classes=5)

# ---- Compile ----
patch_unet.compile(
    optimizer   = Adam(learning_rate=LR),
    loss        = total_loss,
    metrics     = [
        dice_coef,
        DicePerClass(0, name='dice_bg'),
        DicePerClass(1, name='dice_ET'),
        DicePerClass(2, name='dice_NETC'),
        DicePerClass(3, name='dice_SNFH'),
        DicePerClass(4, name='dice_RC'),
        DiceWholeTumorMetric(),
    ],
    run_eagerly = False
)

print('Input  shape:', patch_unet.input_shape)   # (None, 32, 32, 32, 4)
print('Output shape:', patch_unet.output_shape)  # (None, 32, 32, 32, 5)

# ---- Steps per epoch ----
# Each patient gives top_k patches. With batch_size=1, steps = num_patients
steps_per_epoch     = max(1, len(train_patient_list) // BATCH_SIZE)
val_steps_per_epoch = max(1, len(val_patient_list)   // BATCH_SIZE)

# ---- Callbacks ----
os.makedirs('checkpoints', exist_ok=True)

callbacks = [
    ModelCheckpoint(
        filepath       = 'checkpoints/best_abmil_unet.keras',
        monitor        = 'val_dice_whole_tumor',
        mode           = 'max',
        save_best_only = True,
        verbose        = 1
    ),
    EarlyStopping(
        monitor              = 'val_dice_whole_tumor',
        mode                 = 'max',
        patience             = 20,
        restore_best_weights = True,
        verbose              = 1
    ),
    ReduceLROnPlateau(
        monitor  = 'val_dice_whole_tumor',
        factor   = 0.5,
        patience = 10,
        min_lr   = 1e-6,
        mode     = 'max',
        verbose  = 1
    ),
    CSVLogger('training_abmil_unet.log', append=False),
]

# ---- Train ----
history = patch_unet.fit(
    train_generator,
    steps_per_epoch  = steps_per_epoch,
    epochs           = EPOCHS,
    validation_data  = val_generator,
    validation_steps = val_steps_per_epoch,
    callbacks        = callbacks,
    verbose          = 1
)

# ---- Save final model ----
patch_unet.save('model_abmil_unet_brats2024.keras')
print('Training complete. Model saved.')

## Cell 12 — Visualize Results

Show which patches ABMIL found most suspicious and what U-Net segmented.

In [ ]:
def visualize_abmil_unet(volume, mask, abmil_model, unet_model,
                          patch_size=32, top_k=10, device='cpu'):
    """
    For one patient scan:
    1. Extract patches
    2. Show ABMIL attention weights (which patches are suspicious)
    3. Show U-Net segmentation on top patches
    """
    # Step 1: Get patches
    mask_cat = to_categorical(mask, num_classes=5)
    patches, mask_patches = extract_patches(volume, mask_cat, patch_size)

    # Step 2: Get ABMIL weights
    patches_torch = torch.tensor(
        patches.transpose(0, 4, 1, 2, 3), dtype=torch.float32
    ).to(device)

    abmil_model.eval()
    with torch.no_grad():
        weights = abmil_model(patches_torch).cpu().numpy()

    # Step 3: Plot attention weights across all 64 patches
    plt.figure(figsize=(14, 4))
    plt.bar(range(len(weights)), weights, color='steelblue')
    top_indices = np.argsort(weights)[::-1][:top_k]
    plt.bar(top_indices, weights[top_indices], color='tomato', label=f'Top {top_k} patches')
    plt.xlabel('Patch Index')
    plt.ylabel('Attention Weight')
    plt.title('ABMIL Attention Weights — Red = Selected for U-Net')
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Step 4: Show U-Net prediction on top patches
    top_patches = patches[top_indices]   # (top_k, 32, 32, 32, 4)
    preds = unet_model.predict(top_patches, verbose=0)  # (top_k, 32, 32, 32, 5)
    pred_labels = np.argmax(preds, axis=-1)              # (top_k, 32, 32, 32)

    fig, axes = plt.subplots(2, 5, figsize=(18, 7))
    for i in range(min(top_k, 10)):
        mid = patch_size // 2
        ax  = axes[i // 5][i % 5]
        ax.imshow(pred_labels[i, mid, :, :], cmap='jet', vmin=0, vmax=4)
        ax.set_title(f'Patch {top_indices[i]}\nWeight={weights[top_indices[i]]:.3f}')
        ax.axis('off')

    plt.suptitle('U-Net Segmentation on Top ABMIL Patches', fontsize=14)
    plt.tight_layout()
    plt.show()


print('Visualization function defined.')

## Cell 13 — Full Shape Summary

A complete reference of every tensor shape in the pipeline.

In [ ]:
print('=' * 60)
print('COMPLETE SHAPE FLOW SUMMARY')
print('=' * 60)
print()
print('INPUT')
print('  Raw MRI scan + mask')
print('  volume  → (128, 128, 128, 4)')
print('  mask    → (128, 128, 128, 5)')
print()
print('STEP 1 — Patch Extraction')
print('  patches      → (64, 32, 32, 32, 4)')
print('  mask_patches → (64, 32, 32, 32, 5)')
print()
print('STEP 2 — ABMIL Feature Extraction')
print('  Input to CNN        → (64, 4, 32, 32, 32)  [channels first for PyTorch]')
print('  After CNN encoder   → (64, 128)')
print('  After attention     → weights (64,)         [one score per patch]')
print()
print('STEP 3 — Patch Selection')
print('  top_patches      → (10, 32, 32, 32, 4)  [only suspicious patches]')
print('  top_mask_patches → (10, 32, 32, 32, 5)')
print()
print('STEP 4 — U-Net Segmentation')
print('  U-Net input   → (10, 32, 32, 32, 4)')
print('  U-Net output  → (10, 32, 32, 32, 5)  [tumor map per patch]')
print()
print('=' * 60)